# Probeprüfung AP02: Datenbeschaffung, Datenaufbereitung & explorative Datenanalyse (Probeprüfung)

Aufgebaut wie die Prüfung *AP02* — gleiche Teile, gleiche Punkte, gleiches Bewertungsschema —, aber mit den Daten des
fiktiven Velo-Shops **VeloMarkt** statt PhoneMarkt.

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

| Teil | Thema | Aufgaben | Punkte |
|---|---|---|---|
| A | Konzeptfragen: Datenaufbereitung & explorative Datenanalyse | A1–A3 | 12 |
| B | Fehleranalyse Python-Code (Shop-Daten) | B1–B2 | 16 |
| C | Python-Code schreiben: Web Scraping, Datenaufbereitung & EDA | C1–C3 | 12 |
| **Total** | | **8 Aufgaben** | **40** |

### Bewertungsschema

| Kategorie | Beschreibung | Bewertung | 4 Pkt | 8 Pkt |
|---|---|---|---|---|
| Korrekt | Vollständig und fachlich einwandfrei; Code liefert das richtige Ergebnis | 100% | 4 | 8 |
| Minimale Mängel | Kleine Fehler (z. B. falsche Sortierung, fehlende Rundung, fehlende Achsenbeschriftung) | 75% | 3 | 6 |
| Mittlere Mängel | Grössere Fehler (z. B. fehlende Spalten, falsche Filterung) oder Begründung fehlt | 50% | 2 | 4 |
| Gravierende Mängel | Ansatz erkennbar, aber im Kern falsch (z. B. falscher Join-Typ, falsche Datei) | 25% | 1 | 2 |
| Nicht lauffähig | Antwort fehlt oder Code nicht lauffähig | 0% | 0 | 0 |

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege gelten als korrekt.

*Empfohlene Bearbeitungszeit: 60 Minuten. Open book, aber ohne KI — wie in der Prüfung.*

## Hinweise zu den Daten

Alle Daten stammen vom **fiktiven** Schweizer Online-Shop **«VeloMarkt»** und liegen lokal in `Data/velomarkt/`.
Es ist kein Internetzugriff nötig. Spaltenbeschreibungen: `Data/velomarkt/README.md`.

| Datei | Format | Inhalt |
|---|---|---|
| `html/shop_page_1.html` … `shop_page_3.html` | HTML | gespeicherte Produktlisten-Seiten (Paginierung) |
| `html/product_detail.html` | HTML | Detailseite eines E-Bikes (Preisverlauf, technische Daten, Lieferung) |
| `reviews.json` | JSON | gespeicherte Antwort der Reviews-API: Bewertungen pro `product_id` |
| `velos_clean.csv` | CSV | aufbereitete Shop-Daten — **inhaltlich nicht geprüft** |

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import re
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Daten von «VeloMarkt» liegen hier:
DATA = 'Data/velomarkt'
print(os.getcwd())

### Produktlisten-Seiten scrapen → `df_raw`

Der folgende Code liest die drei gespeicherten Produktlisten-Seiten mit BeautifulSoup ein. Jede Produktkarte
(`<div class="product-card">`) wird zu einer Zeile im DataFrame `df_raw`. Die Werte werden **als Text** übernommen
(Spalten mit Endung `_raw`) und noch nicht bereinigt.

In [ ]:
# Initialize list for the extracted rows
rows = []

# Loop over the three listing pages
for page in [1, 2, 3]:

    # Read html from file
    with open(f'{DATA}/html/shop_page_{page}.html', 'r', encoding='utf-8') as file:
        html_content = file.read()

    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Locate the product cards and extract values
    for card in soup.select('div.product-card'):
        row_data = [card['data-product-id'],
                    card.select('span.brand')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.model')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.category')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.weight')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.battery')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.range')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.price')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.availability')[0].get_text().replace('\xa0', '').strip()]
        rows.append(row_data)

# Create a DataFrame
df_raw = pd.DataFrame(rows, columns=['product_id', 'brand', 'model', 'category', 'weight_raw',
                                     'battery_raw', 'range_raw', 'price_raw', 'availability'])

# Show dimensions and first rows
print('Number of rows and columns:', df_raw.shape)
df_raw.head()

### Aufbereitete Shop-Daten laden → `df_clean`

In [ ]:
# Import data from csv to pandas dataframe named 'df_clean'
df_clean = pd.read_csv(f'{DATA}/velos_clean.csv')

# Show dimensions and first rows
print('Number of rows and columns:', df_clean.shape)
df_clean.head()

> **Hinweis:** Alle Aufgaben sind voneinander unabhängig. Verändern Sie `df_raw` und `df_clean` bitte nicht direkt,
> sondern arbeiten Sie in jeder Aufgabe mit einem eigenen DataFrame (z. B. `df_B1`, `df_C2`).

---
**▶ Ab hier beginnt die bewertete Probeprüfung — 40 Punkte**

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den Shop-Daten von «VeloMarkt» werden nicht gewertet.*

### A1 (4 Punkte)

In `velos_clean.csv` fehlt `weight_kg` bei 4 Velos: Der Lieferant hat für einzelne, zufällig verteilte Modelle kein
Datenblatt geliefert. `battery_wh` fehlt bei 59 der 120 Velos — es sind genau die Velos ohne Elektromotor
(Rennvelo, Gravel, City, Kindervelo).

a) Um welchen Mechanismus fehlender Werte handelt es sich jeweils (MCAR, MAR oder MNAR)? Begründen Sie.

b) Ein Analyst ersetzt alle fehlenden `battery_wh` durch den Mittelwert (707 Wh) und berichtet anschliessend die
durchschnittliche Akkugrösse **aller** Velos. Beurteilen Sie dieses Vorgehen.

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Für die Spalte `weight_kg` in `velos_clean.csv` liefert pandas folgendes Ergebnis:

```
df_clean['weight_kg'].describe()          df_clean['weight_kg'].skew()
count    116.00                            0.00
mean      17.30
std        7.55
min        6.80
25%        9.70
50%       16.40
75%       24.42
max       28.50
```

Eine Kollegin schreibt in den Bericht: *«Das typische Velo im Sortiment wiegt rund 17 kg; die Gewichte sind
symmetrisch verteilt.»* Beurteilen Sie diese Aussage anhand der Kennzahlen und nennen Sie eine geeignete Grafik, um Ihre
Einschätzung zu überprüfen.

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Ein Kollege berechnet für alle E-Bikes mit Preisangabe (n = 55) die Korrelation zwischen Akkugrösse und Preis:

```
df[['battery_wh', 'price_chf']].corr()     # Ergebnis: r = -0.10
```

Er folgert: *«Die Akkugrösse hat keinen Einfluss auf den Preis — grössere Akkus muss man im Marketing nicht bewerben.»*

Nennen und begründen Sie **zwei** fachliche Einwände gegen diese Vorgehensweise bzw. Schlussfolgerung.

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (16 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Kennzahlen für das Sortiment aus den gescrapten Rohdaten `df_raw`: Anzahl Velos, Anzahl Velos ohne
Preisangabe und Median-Preis in CHF.

```python
# Select columns for the analysis
df_B1 = df_raw[['product_id', 'brand', 'model', 'category', 'price_raw']]

# Extract numerical values from price_raw
price = []
for i in df_B1['price_raw']:
    d1 = re.findall(r"\d+", i)
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df_B1['price_chf'] = pd.Series(price, dtype="Int64")

# Show key figures
print('Anzahl Velos:             ', df_B1.shape[0])
print('Anzahl ohne Preisangabe:  ', df_B1['price_chf'].isnull().sum())
print('Median-Preis (CHF):       ', df_B1['price_chf'].median())
```

Tatsächliches Ergebnis:

```
Anzahl Velos:              126
Anzahl ohne Preisangabe:   7
Median-Preis (CHF):        4.0
```

Der Shop gibt auf jeder Seite an, dass er **120 Artikel** führt. Beispiele für Werte in `price_raw`: `CHF899.–`,
`CHF3'898.–`, `Preis auf Anfrage`.

**Aufgabe:** Schreiben Sie den korrigierten Code (die Kommentare des ursprünglichen Codes sind als Gerüst vorgegeben),
sodass die Kennzahlen stimmen. Begründen Sie jede Änderung gegenüber dem ursprünglichen Code.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Select columns for the analysis

# Extract numerical values from price_raw

# Save as new variable in the pandas data frame

# Show key figures

### B2 (8 Punkte)

**Ziel:** Herausfinden, welche Marke im Sortiment am teuersten bzw. am günstigsten ist (auf Basis von `df_clean`).

```python
# Select columns for the analysis
df_B2 = df_clean[['brand', 'price_chf']]

# Price per brand
df_B2.groupby(['brand']).mean().round(2).sort_values(by='price_chf', ascending=False)
```

Tatsächliches Ergebnis:

```
                price_chf
brand
Stromer           7849.20
Specialized       5518.00
Scott             4789.67
Flyer             4498.00
Canyon            4356.33
Tour de Suisse    4113.33
Trek              2540.86
Cube              2296.82
```

Der Einkaufsleiter ist skeptisch: *«Tour de Suisse ist unsere Einstiegsmarke mit vielen günstigen City-Velos — teurer
als Trek und Cube kann nicht stimmen.»*

**Aufgabe:** Untersuchen Sie die Daten mit geeigneten EDA-Methoden, finden Sie die Ursache für das irreführende Ergebnis
und schreiben Sie den korrigierten Code. Begründen Sie jede Änderung gegenüber dem ursprünglichen Code.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Select columns for the analysis

# Price per brand

---
## Teil C – Python-Code schreiben (12 Punkte)

Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.

### C1 (4 Punkte)

Die Datei `html/product_detail.html` enthält die Detailseite des Flyer Upstreet 5.23. Das Gerüst unten liest die
HTML-Datei ein und überführt eine Tabelle Zeile für Zeile in den DataFrame `df_C1` mit den Spalten `merkmal` und `wert`.

Ergänzen Sie die beiden mit `# TODO` markierten Stellen:

1. Wählen Sie die Tabelle **«Technische Daten»** aus. Alle drei Tabellen der Seite haben dieselbe Klasse.
2. Entfernen Sie die Zwischenüberschriften der Tabelle (z. B. «Antrieb»). Sie sind **keine** Merkmale und dürfen im
   Ergebnis nicht vorkommen.

Begründen Sie, wie Sie die richtige Tabelle auswählen und die Zwischenüberschriften entfernen.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**Python-Code:**

In [ ]:
# Read html from file
with open(f'{DATA}/html/product_detail.html', 'r', encoding='utf-8') as file:
    html_content = file.read()

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# TODO: Locate the table «Technische Daten»
table = ...

# Extract table rows
rows = []
for row in table.find_all('tr'):
    rows.append([cell.get_text().strip() for cell in row.find_all('td')])

# Create a DataFrame
df_C1 = pd.DataFrame(rows, columns=['merkmal', 'wert'])

# TODO: Remove the sub-headings (e.g. «Antrieb»)


# Show the result
df_C1

### C2 (4 Punkte)

Teilen Sie die Velos in `df_clean` anhand von `price_chf` mit `pd.qcut()` in **vier gleich grosse** Preisklassen mit
den Labels `Einstieg`, `Mittel`, `Gehoben` und `Premium` ein (neue Spalte `segment`).

Erstellen Sie anschliessend mit `pd.crosstab()` eine Kontingenztabelle mit der Anzahl Velos pro Kategorie
(`category`, Zeilen) und Preisklasse (`segment`, Spalten) inklusive Summen (`margins=True`). Begründen Sie, wie
`pd.qcut()` die Klassengrenzen bestimmt und was mit den Velos ohne Preis geschieht.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**Python-Code:**

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Lesen Sie die Kundenbewertungen aus `reviews.json` in einen DataFrame ein und verbinden Sie ihn mit `.merge()` über
`product_id` mit `df_clean`. Berücksichtigen Sie nur Velos mit **mindestens 5 Bewertungen** (`review_count`).
Berechnen Sie die **Spearman-Rangkorrelation** zwischen `price_chf` und `avg_score` (gerundet auf 2 Dezimalstellen) und
erstellen Sie ein Streudiagramm (Scatter Plot) mit beschrifteten Achsen.

Begründen Sie, welche Velos in die Berechnung eingehen, und interpretieren Sie das Ergebnis in 1–2 Sätzen.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**Python-Code:**

In [ ]:
# Code hier eintragen

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')